# 02 · Second and third models: CLIP and DINOv2 indices

**What this notebook does.** Rebuilds the gender-balanced standardization index of the 105 K-pop videos with two other representation models
(CLIP ViT-B/16, an image–text model; DINOv2-base, a self-supervised image model) using the same windows (8 s, stride 4 s, 8 frames) and the same
balanced reference; measures agreement of the three models (Spearman, top/bottom-10 overlap, Cronbach and Krippendorff alpha, ICC) and an
exploratory combined index; scores the Korean indie corpus on the K-pop scale of each model; and scores **new** music videos with all three models.

**Run order.** Notebook 01 must have produced `tables/FINAL_product_results.xlsx` (sheet `ranking`) and `per_video/`. Run sections A → B → C → D.
Section D re-uses the InternVideo2 functions of notebook 01 (copied here unchanged).

**Thesis.** 4-3-5 (Table 4-7) and 4-8-2; appendix C (Table C-2).

---

## ۰۲ · مدل دوم و سوم: شاخص با CLIP و DINOv2
شاخص متوازن ۱۰۵ ویدئو با دو مدل دیگر و همان پنجره‌ها دوباره ساخته می‌شود؛ هم‌نظری سه مدل و شاخص ترکیبی اکتشافی؛ پیکره‌ی ایندی روی مقیاس کی‌پاپ هر مدل؛ و نمره‌دهی ویدئوهای تازه با هر سه مدل. پیش‌نیاز: خروجی‌های نوتبوک ۰۱.

### 1. Install libraries

**فارسی:** نصب کتابخانه‌ها.

*Thesis: appendix C*

In [ ]:
# 1. Install libraries
# Thesis: appendix C
%pip install -q "transformers==4.51.3" "huggingface_hub<1.0" decord openpyxl xlsxwriter pandas numpy tqdm scenedetect librosa open_clip_torch timm einops matplotlib scipy statsmodels psutil imageio-ffmpeg opencv-python pingouin

### 2. PATHS (the only cell to edit)

**فارسی:** مسیرها: تنها خانه‌ای که باید ویرایش شود.

*Thesis: 3-10*

In [ ]:
# 2. PATHS
# Thesis: 3-10
from pathlib import Path
BASE        = Path.home() / 'KPOP'                       # same layout as notebook 01
KPOP_DIR    = BASE / 'kpop Corpus'                       # 105 K-pop videos
INDIE_DIR   = BASE / 'Kindie Corp' if (BASE / 'Kindie Corp').exists() else BASE / 'kindie_30'   # 33 Korean indie videos
METADATA    = BASE / 'MV_metadata_105_verified_v2.xlsx' # corpus metadata (sheet 'Metadata_105')
OUT         = BASE / 'analysis_v2_0'                     # output folder of notebook 01
NEW_DIR     = BASE / 'new_videos'                        # new music videos to score (section D)
VIDEO_DIR   = KPOP_DIR
RESULTS     = OUT / 'tables' / 'FINAL_product_results.xlsx'   # from notebook 01 (sheet 'ranking': file_name, gender, index_balanced_z)
MODEL_DIR   = OUT / 'tables'
CLIP_OUT_FILE  = MODEL_DIR / 'second_model_index.xlsx'
CLIP_VEC_FILE  = MODEL_DIR / 'clip_video_vectors.npy'
DINO_OUT_FILE  = MODEL_DIR / 'third_model_index.xlsx'
DINO_VEC_FILE  = MODEL_DIR / 'dino_video_vectors.npy'
REF_FILE       = MODEL_DIR / 'reference_105.npz'        # all 105 vectors per model + gender + z constants (for new videos)
INDIE_OUT_FILE = MODEL_DIR / 'indie_three_models.xlsx'
INDIE_VEC_FILE = MODEL_DIR / 'indie_vectors_clip_dino.npz'

VIDEO_EXT = ('.mp4', '.mkv', '.webm', '.mov')
def list_videos(d):
    # the high-resolution copies for the eye-tracker (sub-folder '1080') are not part of the corpus
    return sorted(p for p in Path(d).rglob('*') if p.suffix.lower() in VIDEO_EXT and not p.name.startswith('.') and '1080' not in p.parent.name)
for d in ['per_video', 'tables', 'figures']:
    (OUT / d).mkdir(parents=True, exist_ok=True)
print('results file found:', RESULTS.exists(), '| K-pop folder:', KPOP_DIR.exists(), '| indie folder:', INDIE_DIR.exists())

## A. CLIP ViT-B/16

Only the model changes: same 8-s windows, 4-s stride, 8 uniform frames, 224×224, mean pooling, balanced reference, z. CLIP is an image model, so the 8 frames of a window are embedded one by one and averaged.

مدل CLIP: فقط مدل عوض می‌شود؛ فریم‌های هر پنجره جدا بازنمایی و میانگین‌گیری می‌شوند.

### 3. Load CLIP and the ranking sheet

**فارسی:** بارگذاری CLIP و جدول رتبه‌بندی نوتبوک ۰۱.

*Thesis: 4-3-5*

In [ ]:
# 3. Load CLIP ViT-B/16 (open_clip, OpenAI weights) and the InternVideo2 ranking
# Thesis: 4-3-5
import os, numpy as np, pandas as pd, cv2, torch, open_clip
from scipy.stats import spearmanr
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, _, clip_prep = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai'); clip_model = clip_model.to(dev).eval()
print('device:', dev)
R = pd.read_excel(RESULTS, sheet_name='ranking')
missing = [f for f in R.file_name if not os.path.exists(os.path.join(VIDEO_DIR, f))]
print('videos in ranking:', len(R), '| missing files:', len(missing)); print(missing[:10])

### 4. CLIP video vectors

**فارسی:** بردار ویدئوها با CLIP (پنجره‌های ۸ ثانیه‌ای).

*Thesis: 4-3-5*

In [ ]:
# 4. CLIP video vectors (8-s windows, 4-s stride, 8 frames) -> clip_video_vectors.npy
# Thesis: 4-3-5
from PIL import Image
WIN, STRIDE, NF = 8.0, 4.0, 8
def clip_video_vector(path):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 25; n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); dur = n / fps
    starts = np.arange(0, max(dur - WIN, 0) + 1e-6, STRIDE); wins = []
    for s in starts:
        idx = np.linspace(s * fps, (s + WIN) * fps - 1, NF).astype(int); ims = []
        for i in idx:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(min(i, n - 1))); ok, fr = cap.read()
            if ok: ims.append(clip_prep(Image.fromarray(cv2.cvtColor(cv2.resize(fr, (224, 224)), cv2.COLOR_BGR2RGB))))
        if not ims: continue
        with torch.no_grad():
            e = clip_model.encode_image(torch.stack(ims).to(dev)).float(); e = e / e.norm(dim=-1, keepdim=True)
        w = e.mean(0); wins.append((w / w.norm()).cpu().numpy())
    cap.release(); v = np.mean(wins, 0); return v / np.linalg.norm(v), len(wins)
EMB_CLIP, NW = {}, {}
for k, f in enumerate(R.file_name):
    p = os.path.join(VIDEO_DIR, f)
    if not os.path.exists(p): continue
    EMB_CLIP[f], NW[f] = clip_video_vector(p)
    if k % 10 == 0: print(k, f, NW[f], 'windows')
np.save(CLIP_VEC_FILE, np.stack([EMB_CLIP[f] for f in EMB_CLIP])); print('done', len(EMB_CLIP))

### 5. CLIP balanced index and agreement with InternVideo2

**فارسی:** شاخص متوازن با CLIP و هم‌نظری با InternVideo2.

*Thesis: 4-3-5; Table 4-7*

In [ ]:
# 5. Balanced index with CLIP, exactly as for InternVideo2 -> second_model_index.xlsx
# Thesis: 4-3-5; Table 4-7
D = R[R.file_name.isin(EMB_CLIP)].reset_index(drop=True)
X = np.stack([EMB_CLIP[f] for f in D.file_name]); S = X @ X.T; np.fill_diagonal(S, np.nan)
fem = (D.gender.str.lower() == 'female').to_numpy()
bal = 0.5 * np.nanmean(S[:, fem], 1) + 0.5 * np.nanmean(S[:, ~fem], 1)
D['index_clip_z'] = (bal - bal.mean()) / bal.std()
rho, p = spearmanr(D.index_clip_z, D.index_balanced_z)
boot = [spearmanr(*D[['index_clip_z', 'index_balanced_z']].sample(len(D), replace=True, random_state=b).T.to_numpy())[0] for b in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
top = lambda c: set(D.nlargest(10, c).file_name); bot = lambda c: set(D.nsmallest(10, c).file_name)
rf = spearmanr(D.loc[fem, 'index_clip_z'], D.loc[fem, 'index_balanced_z'])[0]; rm = spearmanr(D.loc[~fem, 'index_clip_z'], D.loc[~fem, 'index_balanced_z'])[0]
SUM = pd.DataFrame([dict(n=len(D), rho_clip_vs_internvideo2=rho, ci_low=lo, ci_high=hi, p=p, rho_female=rf, rho_male=rm,
                         top10_overlap=len(top('index_clip_z') & top('index_balanced_z')), bottom10_overlap=len(bot('index_clip_z') & bot('index_balanced_z')))])
display(SUM.round(3))
with pd.ExcelWriter(CLIP_OUT_FILE) as xw:
    SUM.to_excel(xw, sheet_name='summary', index=False)
    D[['file_name', 'artist', 'gender', 'index_balanced_z', 'index_clip_z']].to_excel(xw, sheet_name='per_video', index=False)
print('saved', CLIP_OUT_FILE)

## B. DINOv2-base and three-model agreement

DINOv2 (Oquab et al., 2023) is trained without text, so it sees form only: composition, bodies, light, setting. Same pipeline as above.

مدل DINOv2 بدون متن آموزش دیده و فقط فرم را می‌بیند؛ همان مسیر پردازش.

### 6. Load DINOv2

**فارسی:** بارگذاری DINOv2.

*Thesis: 4-3-5*

In [ ]:
# 6. Load DINOv2-base (facebook/dinov2-base) and the ranking
# Thesis: 4-3-5
import os, numpy as np, pandas as pd, cv2, torch
from PIL import Image
from scipy.stats import spearmanr
from transformers import AutoImageProcessor, AutoModel
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
dino_proc = AutoImageProcessor.from_pretrained('facebook/dinov2-base'); dino_model = AutoModel.from_pretrained('facebook/dinov2-base').to(dev).eval()
print('device:', dev)
R = pd.read_excel(RESULTS, sheet_name='ranking')
missing = [f for f in R.file_name if not os.path.exists(os.path.join(VIDEO_DIR, f))]
print('videos in ranking:', len(R), '| missing files:', len(missing)); print(missing[:10])

### 7. DINOv2 video vectors

**فارسی:** بردار ویدئوها با DINOv2 (توکن CLS).

*Thesis: 4-3-5*

In [ ]:
# 7. DINOv2 video vectors (CLS token, same windows) -> dino_video_vectors.npy
# Thesis: 4-3-5
WIN, STRIDE, NF = 8.0, 4.0, 8
def embed_frames(ims):
    with torch.no_grad():
        x = dino_proc(images=ims, return_tensors='pt').to(dev)
        e = dino_model(**x).pooler_output.float()          # CLS token after the final layer norm
    return torch.nn.functional.normalize(e, dim=-1)
def dino_video_vector(path):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 25; n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); dur = n / fps
    wins = []
    for s in np.arange(0, max(dur - WIN, 0) + 1e-6, STRIDE):
        ims = []
        for i in np.linspace(s * fps, (s + WIN) * fps - 1, NF).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(min(i, n - 1))); ok, fr = cap.read()
            if ok: ims.append(Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
        if not ims: continue
        w = embed_frames(ims).mean(0); wins.append((w / w.norm()).cpu().numpy())
    cap.release(); v = np.mean(wins, 0); return v / np.linalg.norm(v), len(wins)
EMB_DINO = {}
for k, f in enumerate(R.file_name):
    p = os.path.join(VIDEO_DIR, f)
    if not os.path.exists(p): continue
    EMB_DINO[f], nw = dino_video_vector(p)
    if k % 10 == 0: print(k, f, nw, 'windows')
np.save(DINO_VEC_FILE, np.stack([EMB_DINO[f] for f in R.file_name if f in EMB_DINO])); print('done', len(EMB_DINO))

### 8. DINOv2 index and agreement of the three models

**فارسی:** شاخص DINOv2 و هم‌نظری سه مدل.

*Thesis: 4-3-5; Table 4-7*

In [ ]:
# 8. DINOv2 balanced index and agreement with InternVideo2 and CLIP
# Thesis: 4-3-5; Table 4-7
D = R[R.file_name.isin(EMB_DINO)].reset_index(drop=True)
fem = (D.gender.str.lower() == 'female').to_numpy()
def bal_index(X):
    S = X @ X.T; np.fill_diagonal(S, np.nan); b = 0.5 * np.nanmean(S[:, fem], 1) + 0.5 * np.nanmean(S[:, ~fem], 1); return b, (b - b.mean()) / b.std(ddof=1)
Xd = np.stack([EMB_DINO[f] for f in D.file_name]); bal_d, D['index_dino_z'] = bal_index(Xd)
C = pd.read_excel(CLIP_OUT_FILE, sheet_name='per_video')[['file_name', 'index_clip_z']]; D = D.merge(C, on='file_name', how='left')
def agree(a, b):
    r, p = spearmanr(D[a], D[b]); bs = [spearmanr(*D[[a, b]].sample(len(D), replace=True, random_state=k).T.to_numpy())[0] for k in range(2000)]
    t = lambda c: set(D.nlargest(10, c).file_name); u = lambda c: set(D.nsmallest(10, c).file_name)
    return dict(pair=f'{a} vs {b}', rho=r, ci_low=np.percentile(bs, 2.5), ci_high=np.percentile(bs, 97.5), p=p,
                rho_female=spearmanr(D.loc[fem, a], D.loc[fem, b])[0], rho_male=spearmanr(D.loc[~fem, a], D.loc[~fem, b])[0],
                top10_overlap=len(t(a) & t(b)), bottom10_overlap=len(u(a) & u(b)))
AG = pd.DataFrame([agree('index_dino_z', 'index_balanced_z'), agree('index_dino_z', 'index_clip_z'), agree('index_clip_z', 'index_balanced_z')])
display(AG.round(3))

### 9. Combined index (exploratory) and internal consistency

**فارسی:** شاخص ترکیبی (اکتشافی) و همسازی درونی سه مدل.

*Thesis: 4-3-5; Table 4-7*

In [ ]:
# 9. Combined index: mean of the three z-scores, z-scored again (equal weight; exploratory, the registered index stays InternVideo2)
# Thesis: 4-3-5; Table 4-7
Z3 = D[['index_balanced_z', 'index_clip_z', 'index_dino_z']]
comb_raw = Z3.mean(1); D['index_combined_z'] = (comb_raw - comb_raw.mean()) / comb_raw.std(ddof=1)
# internal consistency of the three models (Cronbach alpha on the z-scores) and agreement of each with the combined index
k = 3; cov = Z3.cov().to_numpy(); alpha = k / (k - 1) * (1 - np.trace(cov) / cov.sum())
def kalpha_interval(M):   # Krippendorff alpha, interval level; rows = videos, columns = models as 'raters'
    M = np.asarray(M, float); n_u, m = M.shape; v = M.ravel(); Do_ = np.mean([np.mean((r[:, None] - r[None, :])[~np.eye(m, dtype=bool)] ** 2) for r in M])
    De_ = np.mean((v[:, None] - v[None, :]) ** 2) * len(v) / (len(v) - 1); return 1 - Do_ / De_
icc = None
try:
    import pingouin as pg
    L = Z3.reset_index().melt(id_vars='index', var_name='model', value_name='z'); icc = pg.intraclass_corr(L, targets='index', raters='model', ratings='z').set_index('Type').loc['ICC2', 'ICC']
except Exception: pass
COMB = pd.DataFrame([dict(quantity='Cronbach alpha of the three model indices', value=alpha),
                     dict(quantity='Krippendorff alpha (interval), models as raters', value=kalpha_interval(Z3.to_numpy()))] +
                    ([dict(quantity='ICC(2,1) absolute agreement', value=icc)] if icc is not None else []) +
                    [dict(quantity=f'rho({c}, combined)', value=spearmanr(D[c], D.index_combined_z)[0]) for c in Z3.columns])
# the eight eye-tracking videos: rank in each index (1 = most standardized)
for c in list(Z3.columns) + ['index_combined_z']: D['rank_' + c.replace('index_', '').replace('_z', '')] = D[c].rank(ascending=False).astype(int)
display(COMB.round(3))
display(D.sort_values('index_combined_z', ascending=False)[['file_name', 'gender'] + [c for c in D.columns if c.startswith('rank_')]].head(12))

### 10. Reference file for new videos

**فارسی:** فایل مرجع برای نمره‌دهی ویدئوهای تازه و ذخیره‌ی جدول‌ها.

*Thesis: 4-3-5*

In [ ]:
# 10. Reference file for scoring new videos (all 105 vectors per model + gender + mean/sd used for z) -> reference_105.npz, third_model_index.xlsx
# Thesis: 4-3-5
bal_c = None
if os.path.exists(CLIP_VEC_FILE):
    Xc = np.load(CLIP_VEC_FILE); bal_c, _ = bal_index(Xc)   # same order as the ranking sheet (all 105 found)
np.savez(REF_FILE, file_name=D.file_name.to_numpy(), female=fem, dino=Xd, dino_mean=bal_d.mean(), dino_sd=bal_d.std(ddof=1),
         comb_mean=comb_raw.mean(), comb_sd=comb_raw.std(ddof=1),
         **({'clip': Xc, 'clip_mean': bal_c.mean(), 'clip_sd': bal_c.std(ddof=1)} if bal_c is not None else {}))
with pd.ExcelWriter(DINO_OUT_FILE) as xw:
    AG.to_excel(xw, sheet_name='agreement', index=False); COMB.to_excel(xw, sheet_name='combined', index=False)
    D.drop(columns=[c for c in D.columns if c not in ['file_name', 'artist', 'gender'] and not c.startswith(('index_', 'rank_'))]).to_excel(xw, sheet_name='per_video', index=False)
print('saved', DINO_OUT_FILE, 'and', REF_FILE)

## C. Indie reference corpus with CLIP and DINOv2

Chapter 4 states, with InternVideo2, that the indie corpus is less coherent than K-pop and that no indie video is closer to the K-pop template than the median K-pop video. This section checks both claims with the two other models (same windows and frames).

پیکره‌ی ایندی با دو مدل دیگر: آیا ایندی کم‌انسجام‌تر است و هیچ ویدئوی ایندی از میانه‌ی کی‌پاپ به قالب نزدیک‌تر نیست؟

### 11. Load both models and the indie files

**فارسی:** بارگذاری دو مدل و فهرست ویدئوهای ایندی.

*Thesis: 3-7, 4-8-2*

In [ ]:
# 11. Load CLIP (ViT-B-16-quickgelu, OpenAI weights) and DINOv2, the reference file and the indie video list
# Thesis: 3-7, 4-8-2
import os, numpy as np, pandas as pd, cv2, torch, open_clip
from pathlib import Path
from PIL import Image
from scipy.stats import mannwhitneyu
from transformers import AutoImageProcessor, AutoModel
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
cm, _, cp = open_clip.create_model_and_transforms('ViT-B-16-quickgelu', pretrained='openai'); cm = cm.to(dev).eval()
dp = AutoImageProcessor.from_pretrained('facebook/dinov2-base'); dm = AutoModel.from_pretrained('facebook/dinov2-base').to(dev).eval()
REF = np.load(REF_FILE, allow_pickle=True); fem = REF['female'].astype(bool)
files = sorted(p for p in Path(INDIE_DIR).rglob('*') if p.suffix.lower() in ('.mp4', '.mkv', '.webm', '.mov') and '1080' not in p.parent.name)
print('device:', dev, '| indie videos found:', len(files))

### 12. Indie vectors with CLIP and DINOv2

**فارسی:** بردار ویدئوهای ایندی با هر دو مدل.

*Thesis: 4-8-2*

In [ ]:
# 12. Indie video vectors with CLIP and DINOv2 (identical sampling) -> indie_vectors_clip_dino.npz
# Thesis: 4-8-2
WIN, STR, NF = 8.0, 4.0, 8
@torch.no_grad()
def vectors(path):   # identical sampling to the CLIP and DINOv2 notebooks
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25; n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); wc, wd = [], []
    for s_ in np.arange(0, max(n / fps - WIN, 0) + 1e-6, STR):
        ims = []
        for i in np.linspace(s_ * fps, (s_ + WIN) * fps - 1, NF).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(min(i, n - 1))); ok, fr = cap.read()
            if ok: ims.append(fr)
        if not ims: continue
        e = cm.encode_image(torch.stack([cp(Image.fromarray(cv2.cvtColor(cv2.resize(f, (224, 224)), cv2.COLOR_BGR2RGB))) for f in ims]).to(dev)).float()
        e = e / e.norm(dim=-1, keepdim=True); w = e.mean(0); wc.append((w / w.norm()).cpu().numpy())
        x = dp(images=[Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in ims], return_tensors='pt').to(dev)
        e = torch.nn.functional.normalize(dm(**x).pooler_output.float(), dim=-1); w = e.mean(0); wd.append((w / w.norm()).cpu().numpy())
    cap.release(); vc, vd = np.mean(wc, 0), np.mean(wd, 0); return vc / np.linalg.norm(vc), vd / np.linalg.norm(vd)
IC, ID = [], []
for k, p in enumerate(files):
    c, d = vectors(p); IC.append(c); ID.append(d)
    if k % 5 == 0: print(k, p.name)
IC, ID = np.stack(IC), np.stack(ID); np.savez(INDIE_VEC_FILE, file_name=[p.name for p in files], clip=IC, dino=ID); print('done')

### 13. Indie on the K-pop scale; coherence K-pop vs indie

**فارسی:** ایندی روی مقیاس کی‌پاپ هر مدل و انسجام دو پیکره (یک ویدئو برای هر هنرمند).

*Thesis: 4-8-2*

In [ ]:
# 13. Indie on the K-pop scale of each model and K-pop minus indie coherence (one video per artist) -> indie_three_models.xlsx
# Thesis: 4-8-2
artist_i = np.array([p.name.split(' - ')[0].split('_')[0].replace('[MV]', '').replace('[Official M_V]', '').strip() for p in files])
kp_names = REF['file_name']; artist_k = np.array([str(f).split(' ')[0] for f in kp_names])
def bal_k(sim): return 0.5 * sim[:, fem].mean(1) + 0.5 * sim[:, ~fem].mean(1)
def one_per_artist(a, rng): return pd.Series(range(len(a))).groupby(pd.Series(a)).apply(lambda s: rng.choice(s.values)).to_numpy()
coh = lambda X: (X @ X.T)[np.triu_indices(len(X), 1)].mean()
ROWS, PER = [], {}
for m, XK, XI in [('CLIP', REF['clip'], IC), ('DINOv2', REF['dino'], ID)]:
    mu, sd = float(REF[m.lower().replace('v2', '') + '_mean']), float(REF[m.lower().replace('v2', '') + '_sd'])
    SK = XK @ XK.T; np.fill_diagonal(SK, np.nan)
    kz = (0.5 * np.nanmean(SK[:, fem], 1) + 0.5 * np.nanmean(SK[:, ~fem], 1) - mu) / sd     # K-pop on its own scale
    iz = (bal_k(XI @ XK.T) - mu) / sd                                                       # indie on the same K-pop scale
    PER[m] = iz
    rng = np.random.default_rng(12); dd = []
    for _ in range(2000):
        a = one_per_artist(artist_i, rng); b = one_per_artist(artist_k, rng); n_ = min(len(a), len(b))
        dd.append(coh(XK[rng.choice(b, n_, replace=False)]) - coh(XI[rng.choice(a, n_, replace=False)]))
    ROWS.append(dict(model=m, kpop_median_z=np.median(kz), indie_median_z=np.median(iz), indie_max_z=iz.max(),
                     n_indie_above_kpop_median=int((iz > np.median(kz)).sum()), n_indie=len(iz),
                     mannwhitney_p=mannwhitneyu(kz, iz)[1],
                     coherence_kpop_minus_indie=np.median(dd), ci_low=np.percentile(dd, 2.5), ci_high=np.percentile(dd, 97.5),
                     share_draws_kpop_more_coherent=np.mean(np.array(dd) > 0)))
RES = pd.DataFrame(ROWS); display(RES.round(3))
P = pd.DataFrame(dict(file_name=[p.name for p in files], clip_z_on_kpop_scale=PER['CLIP'], dino_z_on_kpop_scale=PER['DINOv2']))
with pd.ExcelWriter(INDIE_OUT_FILE) as xw:
    RES.to_excel(xw, sheet_name='summary', index=False); P.to_excel(xw, sheet_name='per_indie_video', index=False)
print('saved', INDIE_OUT_FILE)

## D. Scoring new music videos with all three models

This section needs the InternVideo2 functions of notebook 01. They are copied here **unchanged** (steps 14–17) so the notebook is self-contained; then step 18 scores every video in `NEW_DIR`. 0 = like the average K-pop video of the corpus; +1 = one SD more standardized.

نمره‌دهی ویدئوهای تازه با هر سه مدل؛ تابع‌های InternVideo2 بدون تغییر از نوتبوک ۰۱ کپی شده‌اند.

### 14. Machine check (copy of notebook 01, step 1)

**فارسی:** بررسی دستگاه (کپی از نوتبوک ۰۱).

*Thesis: 3-10*

In [ ]:
# 14. Machine check (copied from notebook 01)
# Thesis: 3-10
import os, sys, torch
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
torch.set_num_threads(max(1, (os.cpu_count() or 4) - 2))
print('CPU threads used:', torch.get_num_threads())

### 15. InternVideo2 settings and functions (copy of notebook 01, step 4)

**فارسی:** تنظیمات و توابع InternVideo2 (کپی از نوتبوک ۰۱).

*Thesis: 3-3-1*

In [ ]:
# 15. Fixed InternVideo2 settings and functions (copied unchanged from notebook 01)
# Thesis: 3-3-1
import types, importlib.machinery, re, json, gc, hashlib, math, time, shutil, subprocess, urllib.request
import numpy as np, pandas as pd, cv2
import torch.nn.functional as F
from decord import VideoReader, cpu
from tqdm.auto import tqdm

def install_flash_attn_stub():
    # FlashAttention is disabled in the model config; these empty modules only let the model code import.
    class _Unused:
        def __init__(self, *a, **k): raise RuntimeError('flash_attn stub called: FlashAttention must stay disabled')
    def _getattr(name):
        if name.startswith('__'): raise AttributeError(name)
        return _Unused
    for name in ['flash_attn', 'flash_attn.flash_attn_interface', 'flash_attn.bert_padding', 'flash_attn.modules',
                 'flash_attn.modules.mlp', 'flash_attn.ops', 'flash_attn.ops.rms_norm', 'flash_attn.ops.fused_dense', 'flash_attn.layers',
                 'flash_attn.layers.rotary']:
        m = types.ModuleType(name); m.__path__ = []
        m.__spec__ = importlib.machinery.ModuleSpec(name, loader=None, is_package=True); m.__getattr__ = _getattr
        sys.modules[name] = m
install_flash_attn_stub()

MODEL_ID = 'OpenGVLab/InternVideo2_CLIP_S'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32   # CPU needs float32
NUM_FRAMES, CLIP_SECONDS, STRIDE_SECONDS, EMB_DIM = 8, 8.0, 4.0, 512

def make_clip_starts(duration, clip_seconds=CLIP_SECONDS, stride_seconds=STRIDE_SECONDS):
    # Rule of the FIRST REAL RUN (verified from its checkpoints: 198.24 s -> 50 windows, 186.98 s -> 47 windows):
    # a window starts every 4 s until the end of the video; windows near the end are clipped at the video end.
    return [i * stride_seconds for i in range(max(1, math.ceil(duration / stride_seconds)))]

def sample_video_clip(vr, fps, start_sec, end_sec, num_frames=8):
    n = len(vr)
    start_frame = max(0, int(round(start_sec * fps))); end_frame = min(n, int(round(end_sec * fps)))
    if end_frame <= start_frame:
        raise ValueError('Invalid temporal range.')
    if end_frame - start_frame < num_frames:   # very short last window: repeat frames so the model always gets 8
        idx = np.linspace(start_frame, end_frame - 1, num_frames).round().astype(int).clip(0, n - 1).tolist()
        return torch.from_numpy(vr.get_batch(idx).asnumpy()).permute(0, 3, 1, 2).contiguous()
    k = num_frames
    edges = np.linspace(start_frame, end_frame, k + 1).astype(int)
    idx = [min(int(edges[i]), n - 1) if int(edges[i + 1]) <= int(edges[i]) else (int(edges[i]) + int(edges[i + 1]) - 1) // 2 for i in range(k)]
    return torch.from_numpy(vr.get_batch(idx).asnumpy()).permute(0, 3, 1, 2).contiguous()   # T,C,H,W uint8

def window_frame_indices(n, fps, start_sec, end_sec, num_frames=8):
    start_frame = max(0, int(round(start_sec * fps))); end_frame = min(n, int(round(end_sec * fps)))
    if end_frame <= start_frame: end_frame = min(n, start_frame + 1)
    if end_frame - start_frame < num_frames:
        return np.linspace(start_frame, end_frame - 1, num_frames).round().astype(int).clip(0, n - 1).tolist()
    edges = np.linspace(start_frame, end_frame, num_frames + 1).astype(int)
    return [min(int(edges[i]), n - 1) if int(edges[i + 1]) <= int(edges[i]) else (int(edges[i]) + int(edges[i + 1]) - 1) // 2 for i in range(num_frames)]

_model = None
def get_model():
    global _model
    if _model is None:
        from transformers import AutoModel
        _model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=DTYPE).eval().to(DEVICE)
        print('InternVideo2 loaded')
    return _model

V_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
V_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)

def preprocess(frames):
    # official InternVideo2 demo preprocessing: resize to 224x224, scale to [0,1], ImageNet mean/std, shape [1,T,3,224,224]
    x = frames.float() / 255.0                                            # T,C,H,W
    x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
    return ((x.unsqueeze(0) - V_MEAN) / V_STD)

@torch.inference_mode()
def encode_clip(frames):
    feat = get_model().encode_vision(preprocess(frames).to(DEVICE, DTYPE), test=True).float()
    return F.normalize(feat, dim=-1).reshape(-1).cpu()

def l2n(x, axis=-1):
    return x / np.clip(np.linalg.norm(x, axis=axis, keepdims=True), 1e-12, None)

def slug(name):
    return re.sub(r'[^\w\-.]+', '_', Path(name).stem, flags=re.UNICODE).strip('._')[:90] or 'video'

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

def save_npz_atomic(path, **kw):
    tmp = Path(str(path) + '.tmp.npz'); np.savez_compressed(tmp, **kw); os.replace(tmp, path)

def save_json_atomic(path, obj):
    tmp = Path(str(path) + '.tmp'); tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding='utf-8'); os.replace(tmp, path)
import imageio_ffmpeg
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()   # ffmpeg shipped via pip (no conda needed)
print('ffmpeg:', FFMPEG)
print('functions ready')

### 16. Extraction functions (copy of notebook 01, step 6)

**فارسی:** تابع‌های استخراج (کپی از نوتبوک ۰۱؛ اینجا فقط stage_embeddings به کار می‌رود).

*Thesis: 3-3-1*

In [ ]:
# 16. Per-video pipeline functions (copied unchanged from notebook 01; only stage_embeddings is used here)
# Thesis: 3-3-1
OLD_INDEX = {}   # first-run checkpoints are not needed for new videos
YUNET = OUT / 'face_detection_yunet_2023mar.onnx'
if not YUNET.exists():
    try:
        urllib.request.urlretrieve('https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx', YUNET)
    except Exception as e:
        print('YuNet download failed (Haar fallback will be used):', e)

def face_detector():
    if YUNET.exists():
        det = cv2.FaceDetectorYN.create(str(YUNET), '', (320, 320), 0.6)
        def f(img):
            det.setInputSize((img.shape[1], img.shape[0])); _, faces = det.detect(img)
            return [] if faces is None else [tuple(map(float, fc[:4])) for fc in faces]
        return f, 'yunet'
    haar = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    return (lambda img: [tuple(map(float, b)) for b in haar.detectMultiScale(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 1.1, 5)]), 'haar'

def colorfulness(img):
    b, g, r = [c.astype(np.float32) for c in cv2.split(img)]
    rg, yb = r - g, 0.5 * (r + g) - b
    return float(np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))

def stage_embeddings(path, d, force_new=False):
    out = d / 'embeddings.npz'
    if out.exists() and not force_new: return 'skip'
    if not force_new and path.name in OLD_INDEX:                     # reuse first-run window embeddings
        z = np.load(OLD_INDEX[path.name], allow_pickle=False)
        save_npz_atomic(out, clip_embeddings=z['clip_embeddings'], clip_starts=z['clip_starts'], clip_ends=z['clip_ends'],
                        fps=z['fps'], duration_sec=z['duration_sec'], source='v1.0 checkpoint ' + OLD_INDEX[path.name].name)
        return 'reused v1.0'
    vr = VideoReader(str(path), ctx=cpu(0)); fps = float(vr.get_avg_fps()); n = len(vr); dur = n / fps; del vr; gc.collect()
    wins = [(s_, min(dur, s_ + CLIP_SECONDS)) for s_ in make_clip_starts(dur)]
    win_idx = [window_frame_indices(n, fps, s_, e_, NUM_FRAMES) for s_, e_ in wins]
    need = {}                                      # frame index -> windows that use it
    for w, idx in enumerate(win_idx):
        for j in idx: need.setdefault(j, set()).add(w)
    # memory-light: read the video ONCE sequentially (no decord batch buffers), keep only frames still needed
    cap = cv2.VideoCapture(str(path)); buf = {}; E = [None] * len(wins); nxt = 0; k = 0
    last_needed = max(need)
    while k <= last_needed:
        if k in need:
            ok, fr = cap.read()
            if not ok: break
            buf[k] = torch.from_numpy(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).contiguous()
        else:
            if not cap.grab(): break
        while nxt < len(wins) and all(j in buf for j in win_idx[nxt]):     # window complete -> encode, free frames
            E[nxt] = encode_clip(torch.stack([buf[j] for j in win_idx[nxt]])).numpy().astype(np.float32)
            for j in set(win_idx[nxt]):
                need[j].discard(nxt)
                if not need[j]: buf.pop(j, None)
            nxt += 1
        k += 1
    cap.release()
    while nxt < len(wins):                         # frames past the decodable end: repeat the last decoded frame
        last = buf[max(buf)] if buf else None
        E[nxt] = encode_clip(torch.stack([buf.get(j, last) for j in win_idx[nxt]])).numpy().astype(np.float32); nxt += 1
    save_npz_atomic(out, clip_embeddings=np.stack(E), clip_starts=np.array([w[0] for w in wins], np.float32),
                    clip_ends=np.array([w[1] for w in wins], np.float32), fps=np.float32(fps), duration_sec=np.float32(dur),
                    source='extracted v2.0 (frozen v1.0 settings, sequential cv2 read)')
    buf.clear(); gc.collect()
    return 'extracted'

def stage_visual(path, d, motion_fps=10.0, face_fps=2.0, small_w=320):
    out = d / 'visual.npz'
    if out.exists(): return 'skip'
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    m_step, f_step = max(1, round(fps / motion_fps)), max(1, round(fps / face_fps))
    det, det_name = face_detector()
    t_m, motion, cam = [], [], []
    t_f, lum, con, col, nface, maxface, face_off = [], [], [], [], [], [], []
    prev = None; k = 0; hann = None
    while True:
        ok, fr = cap.read()
        if not ok: break
        if k % m_step == 0:
            small = cv2.resize(fr, (small_w, int(fr.shape[0] * small_w / fr.shape[1])))
            g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            if prev is not None:
                if hann is None: hann = cv2.createHanningWindow(g.shape[::-1], cv2.CV_32F)
                (dx, dy), _ = cv2.phaseCorrelate(prev, g, hann)           # global shift ~ camera pan/tilt
                t_m.append(k / fps); motion.append(float(np.abs(g - prev).mean())); cam.append(float(np.hypot(dx, dy) / small_w))
            prev = g
        if k % f_step == 0:
            small = cv2.resize(fr, (small_w, int(fr.shape[0] * small_w / fr.shape[1])))
            g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            t_f.append(k / fps); lum.append(float(g.mean())); con.append(float(g.std())); col.append(colorfulness(small))
            faces = det(fr); H, W = fr.shape[:2]; nface.append(len(faces))
            if faces:
                areas = [a[2] * a[3] / (H * W) for a in faces]; j = int(np.argmax(areas)); x, y, w, h = faces[j]
                maxface.append(areas[j]); face_off.append(float(np.hypot((x + w / 2) / W - 0.5, (y + h / 2) / H - 0.5)))
            else:
                maxface.append(0.0); face_off.append(np.nan)
        k += 1
    cap.release()
    save_npz_atomic(out, t_motion=np.array(t_m, np.float32), motion=np.array(motion, np.float32), camera_shift=np.array(cam, np.float32),
                    t_frame=np.array(t_f, np.float32), luminance=np.array(lum, np.float32), contrast=np.array(con, np.float32),
                    colorfulness=np.array(col, np.float32), n_faces=np.array(nface, np.int16), max_face_area=np.array(maxface, np.float32),
                    face_centre_offset=np.array(face_off, np.float32), fps=np.float32(fps), n_frames=np.int64(k), face_detector=det_name)
    return 'done'

def stage_shots(path, d):
    out = d / 'shots.json'
    if out.exists(): return 'skip'
    from scenedetect import open_video as sd_open, SceneManager
    from scenedetect.detectors import ContentDetector
    sm = SceneManager(); sm.add_detector(ContentDetector(threshold=27.0)); v = sd_open(str(path)); sm.detect_scenes(v)
    shots = [[s.get_seconds(), e.get_seconds()] for s, e in sm.get_scene_list()]
    save_json_atomic(out, {'detector': 'PySceneDetect ContentDetector threshold=27', 'shots': shots})
    return 'done'

def stage_audio(path, d, sr=22050, hop_sec=0.5):
    out = d / 'audio.npz'
    if out.exists(): return 'skip'
    import librosa
    wav = d / 'audio.wav'
    if not wav.exists():
        subprocess.run([FFMPEG, '-loglevel', 'error', '-y', '-i', str(path), '-ac', '1', '-ar', str(sr), str(wav)], check=True)
    y, _ = librosa.load(wav, sr=sr); hop = int(sr * hop_sec); gc.collect()
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
    save_npz_atomic(out, hop_sec=np.float32(hop_sec), sr=np.int32(sr),
                    chroma=librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop).astype(np.float32),
                    mfcc=librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, hop_length=hop).astype(np.float32),
                    rms=librosa.feature.rms(y=y, hop_length=hop)[0].astype(np.float32),
                    onset=librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop).astype(np.float32),
                    tempo=np.float32(np.atleast_1d(tempo)[0]), beat_times=librosa.frames_to_time(beats, sr=sr).astype(np.float32))
    return 'done'

def stage_contact(path, d, seg=8.0, cols=6, tw=320):
    out = d / 'contact_sheet.jpg'
    if out.exists(): return 'skip'
    from PIL import Image, ImageDraw
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0; n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); dur = n / fps
    times = np.arange(seg / 2, dur, seg); want = {min(n - 1, int(t * fps)): i for i, t in enumerate(times)}; got = {}; k = 0
    while k <= max(want):
        if k in want:
            ok, f_ = cap.read()
            if not ok: break
            got[want[k]] = cv2.cvtColor(f_, cv2.COLOR_BGR2RGB)
        elif not cap.grab(): break
        k += 1
    cap.release(); fr = np.stack([got[i] for i in sorted(got)]); times = times[sorted(got)]
    th = int(fr.shape[1] * tw / fr.shape[2]); rows = math.ceil(len(fr) / cols)
    sheet = Image.new('RGB', (cols * tw, rows * (th + 18)), 'white'); dr = ImageDraw.Draw(sheet)
    for i, (f, t) in enumerate(zip(fr, times)):
        x, y = (i % cols) * tw, (i // cols) * (th + 18); s0 = int(t - seg / 2)
        sheet.paste(Image.fromarray(f).resize((tw, th)), (x, y + 18)); dr.text((x + 4, y + 2), f'seg {i+1:02d}  {s0//60}:{s0%60:02d}', fill='black')
    tmp = d / 'contact_sheet.tmp.jpg'; sheet.save(tmp, quality=80); os.replace(tmp, out)
    del fr, sheet, got; gc.collect()
    return 'done'

STAGES = [('embeddings', stage_embeddings), ('visual', stage_visual), ('shots', stage_shots), ('audio', stage_audio), ('contact', stage_contact)]

def process_corpus(src_dir, corpus, only=None):
    base = OUT / 'per_video' / corpus; base.mkdir(parents=True, exist_ok=True)
    log_path = OUT / 'tables' / f'run_log_{corpus}.csv'
    files = list_videos(src_dir)[: only or None]
    for i, p in enumerate(files):
        d = base / slug(p.name); d.mkdir(exist_ok=True)
        info = d / 'info.json'
        if not info.exists():
            save_json_atomic(info, {'file_name': p.name, 'relative_path': str(p.relative_to(src_dir)), 'sha256': sha256(p), 'size_mb': p.stat().st_size / 2**20})
        status = {}
        for name, fn in STAGES:
            try:
                t0 = time.time(); status[name] = fn(p, d); status[name + '_s'] = round(time.time() - t0, 1)
            except Exception as e:
                status[name] = 'ERROR: ' + repr(e)[:200]
        row = pd.DataFrame([{'time': time.strftime('%Y-%m-%d %H:%M'), 'file_name': p.name, **status}])
        row.to_csv(log_path, mode='a', header=not log_path.exists(), index=False)
        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        import psutil; ram = psutil.virtual_memory().percent
        print(f'[{i+1}/{len(files)}] RAM {ram:.0f}% | {p.name[:55]} ->', {k: v for k, v in status.items() if not k.endswith('_s')})
        if ram > 85:
            print('RAM above 85%: stopping safely. Restart the session (Runtime > Restart session), run steps 1-6, then step 8 again; finished videos are skipped.')
            return
print('pipeline ready')

### 17. Fast load of the saved index

**فارسی:** بارگذاری سریع بردارهای ذخیره‌شده و بازسازی شاخص (همان نوتبوک ۰۳).

*Thesis: 3-3-2*

In [ ]:
# 17. Fast load: saved K-pop and indie embeddings, index rebuilt exactly as in notebook 01 (no figures, no bootstraps)
# Thesis: 3-3-2
from scipy.stats import spearmanr, mannwhitneyu
import unicodedata, re, json

def load_corpus(corpus):
    rows = {}
    for d in sorted((OUT / 'per_video' / corpus).iterdir()):
        if not ((d / 'info.json').exists() and (d / 'embeddings.npz').exists()): continue
        info = json.loads((d / 'info.json').read_text(encoding='utf-8')); e = np.load(d / 'embeddings.npz', allow_pickle=True)
        W = l2n(e['clip_embeddings'].astype(np.float64))
        rows[info['file_name']] = dict(dir=d, W=W, starts=e['clip_starts'], ends=e['clip_ends'], dur=float(e['duration_sec']), v=l2n(W.mean(0)), u=W.mean(0))
    return rows

K = load_corpus('kpop'); Ii = load_corpus('kindie')
meta = pd.read_excel(METADATA, sheet_name='Metadata_105')
def _key(x): return re.sub(r'[^0-9a-z가-힣]+', '', unicodedata.normalize('NFC', str(x)).lower())
_kmap = {_key(k): k for k in K}
meta['file_name'] = [_kmap.get(_key(x), x) for x in meta['file_name']]
meta = meta[meta['file_name'].isin(K)].reset_index(drop=True)
names = list(meta['file_name']); fem = (meta['gender'].str.lower() == 'female').to_numpy()
V = np.stack([K[x]['v'] for x in names]); S = V @ V.T; np.fill_diagonal(S, np.nan)
zs = lambda x: (x - np.nanmean(x)) / np.nanstd(x, ddof=1)
def balanced(Smat, f):
    return 0.5 * np.nanmean(np.where(f[None, :], Smat, np.nan), 1) + 0.5 * np.nanmean(np.where(~f[None, :], Smat, np.nan), 1)
T = meta[['file_name', 'artist', 'song', 'gender', 'release_year']].copy()
T['index_balanced'] = balanced(S, fem); T['index_balanced_z'] = zs(T['index_balanced'])
print('K-pop videos:', len(names), '| indie videos:', len(Ii))

### 18. Score new videos with all three models

**فارسی:** نمره‌ی ویدئوهای تازه با InternVideo2، CLIP، DINOv2 و شاخص ترکیبی در برابر ۱۰۵ ویدئو.

*Thesis: 4-3-5*

In [ ]:
# 18. Score new music videos with InternVideo2 + CLIP + DINOv2 against the 105 K-pop videos -> tables/new_video_scores.xlsx
# Thesis: 4-3-5
import tempfile, open_clip
from PIL import Image
from transformers import AutoImageProcessor, AutoModel as _AM
REF = np.load(REF_FILE, allow_pickle=True)
order = [list(names).index(f) for f in REF['file_name']]                 # put the InternVideo2 vectors in the reference order
Vi = V[order]; fem_r = REF['female'].astype(bool)
Si = Vi @ Vi.T; np.fill_diagonal(Si, np.nan); bal_i = balanced(Si, fem_r)
iv_mean, iv_sd = np.nanmean(bal_i), np.nanstd(bal_i, ddof=1)
check = np.corrcoef(zs(balanced(S, fem)), T.set_index('file_name').index_balanced_z.reindex(names))[0, 1]
print('InternVideo2 reference reproduces index_balanced_z: r =', round(check, 4))

dev_ = 'cuda' if torch.cuda.is_available() else 'cpu'
cm, _, cp = open_clip.create_model_and_transforms('ViT-B-16-quickgelu', pretrained='openai'); cm = cm.to(dev_).eval()
dp = AutoImageProcessor.from_pretrained('facebook/dinov2-base'); dm = _AM.from_pretrained('facebook/dinov2-base').to(dev_).eval()
WIN, STR, NF = 8.0, 4.0, 8
def frame_windows(path):   # same sampling as the CLIP and DINOv2 notebooks
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25; n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); out = []
    for s_ in np.arange(0, max(n / fps - WIN, 0) + 1e-6, STR):
        ims = []
        for i in np.linspace(s_ * fps, (s_ + WIN) * fps - 1, NF).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(min(i, n - 1))); ok, fr = cap.read()
            if ok: ims.append(fr)
        if ims: out.append(ims)
    cap.release(); return out
@torch.no_grad()
def clip_dino_vectors(path):
    wc, wd = [], []
    for ims in frame_windows(path):
        e = cm.encode_image(torch.stack([cp(Image.fromarray(cv2.cvtColor(cv2.resize(f, (224, 224)), cv2.COLOR_BGR2RGB))) for f in ims]).to(dev_)).float()
        e = e / e.norm(dim=-1, keepdim=True); w = e.mean(0); wc.append((w / w.norm()).cpu().numpy())
        x = dp(images=[Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in ims], return_tensors='pt').to(dev_)
        e = torch.nn.functional.normalize(dm(**x).pooler_output.float(), dim=-1); w = e.mean(0); wd.append((w / w.norm()).cpu().numpy())
    vc, vd = np.mean(wc, 0), np.mean(wd, 0); return vc / np.linalg.norm(vc), vd / np.linalg.norm(vd)
def zscore_new(v, X, mu, sd):
    s = X @ v; return ((0.5 * s[fem_r].mean() + 0.5 * s[~fem_r].mean()) - mu) / sd

rows = []
for path in list_videos(NEW_DIR):
    with tempfile.TemporaryDirectory() as td:                    # InternVideo2 exactly as for the corpus (frozen windows and preprocessing)
        stage_embeddings(path, Path(td), force_new=True)
        e = np.load(Path(td) / 'embeddings.npz'); W = l2n(e['clip_embeddings'].astype(np.float64)); vi = l2n(W.mean(0))
    vc, vd = clip_dino_vectors(path)
    z_iv = zscore_new(vi, Vi, iv_mean, iv_sd); z_cl = zscore_new(vc, REF['clip'], float(REF['clip_mean']), float(REF['clip_sd']))
    z_di = zscore_new(vd, REF['dino'], float(REF['dino_mean']), float(REF['dino_sd']))
    z_co = (np.mean([z_iv, z_cl, z_di]) - float(REF['comb_mean'])) / float(REF['comb_sd'])
    rows.append(dict(file_name=path.name, internvideo2_z=z_iv, clip_z=z_cl, dino_z=z_di, combined_z=z_co,
                     percentile_vs_105_internvideo2=float((T.index_balanced_z < z_iv).mean() * 100)))
    print(path.name, {k: round(v, 2) for k, v in rows[-1].items() if k.endswith('_z')})
NEWS = pd.DataFrame(rows); NEWS.to_excel(OUT / 'tables' / 'new_video_scores.xlsx', index=False); display(NEWS.round(2))
print('0 = like the average K-pop video of the corpus; +1 = one SD more standardized; negative = farther from the shared template')